In [1]:
##### Final cleaning of RF training data before runs
# convert to log, remove outliers, add country codes 

import os
import pandas as pd
from pathlib import Path
import numpy as np
import geopandas as gpd
from functools import reduce

In [2]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent.parent 

# data 
capital = pd.read_csv(f'{cd}/Data/Clean/Training_data/capital.csv')
labor = pd.read_csv(f'{cd}/Data/Clean/Training_data/labor.csv')

# save paths
save_path_capital = f'{cd}/Data/Clean/Training_data/capital_final.csv'
save_path_labor = f'{cd}/Data/Clean/Training_data/labor_final.csv'

In [3]:
##### Variables 

### Drop 1 from % variables (so don't sum to 100)
# one of each of field size %'s and production mix %'s 
# share_vlarge_field, other_share_production_USD  

col_to_drop = ['share_vlarge_field', 'other_share_production_USD']

capital = capital.drop(columns = col_to_drop)
labor = labor.drop(columns = col_to_drop)

### Drop aggregate variables

capital = capital.drop(columns = ['GEO_ID_NAME', 'ag_capital_stock_USD_nominal', 'total_production_USD'])
labor = labor.drop(columns = ['GEO_ID_NAME', 'ag_jobs', 'total_production_USD'])

In [4]:
##### Drop regions with missing data 

c_len_pre = len(capital)
capital = capital.dropna()
c_len_pos = len(capital)
c_dropped = c_len_pre - c_len_pos

print(f'Capital: {c_dropped} out of {c_len_pre} dropped')

l_len_pre = len(labor)
labor = labor.dropna()
l_len_pos = len(labor)
l_dropped = l_len_pre - l_len_pos

print(f'Labor: {l_dropped} out of {l_len_pre} dropped')

Capital: 371 out of 10482 dropped
Labor: 469 out of 11500 dropped


In [5]:
#####  Drop extreme values (top/bottom 1%)

lower_cap = capital['capital_intensity_USD_per_USD'].quantile(0.01)
upper_cap = capital['capital_intensity_USD_per_USD'].quantile(0.99)

capital = capital[
    (capital['capital_intensity_USD_per_USD'] >= lower_cap) &
    (capital['capital_intensity_USD_per_USD'] <= upper_cap)
]

lower_lab = labor['labor_intensity_jobs_per_million_USD'].quantile(0.01)
upper_lab = labor['labor_intensity_jobs_per_million_USD'].quantile(0.99)

labor = labor[
    (labor['labor_intensity_jobs_per_million_USD'] >= lower_lab) &
    (labor['labor_intensity_jobs_per_million_USD'] <= upper_lab)
]

In [6]:
##### Add country ID column

capital['country_ID'] = capital['PROJ_ID'].str[:3]
labor['country_ID'] = labor['PROJ_ID'].str[:3]

In [7]:
##### Convert intensities to log

capital["log_capital_intensity_USD_per_USD"] = np.log(capital["capital_intensity_USD_per_USD"])
capital = capital.drop(columns=["capital_intensity_USD_per_USD"])

labor["log_labor_intensity_jobs_per_million_USD"] = np.log(labor["labor_intensity_jobs_per_million_USD"])
labor = labor.drop(columns=["labor_intensity_jobs_per_million_USD"])

In [8]:
##### Save 
capital.to_csv(save_path_capital, index=False)
labor.to_csv(save_path_labor, index=False)